# Q3

In [1]:
import gurobipy as gp

m = gp.Model()
x = m.addVar()
m.setObjective(x)
m.optimize()

Set parameter Username
Set parameter LicenseID to value 2808526
Academic license - for non-commercial use only - expires 2027-04-16
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 23.3.0 23D2057)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 0 rows, 1 columns and 0 nonzeros
Model fingerprint: 0x84abb838
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [0e+00, 0e+00]
Presolve removed 0 rows and 1 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  0.000000000e+00


In [5]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

# --------------------------------------------------
# 1. Create model
# --------------------------------------------------
model3 = gp.Model("Q3")
model3.setParam("OutputFlag", 1)

print("Teams:", len(T))
print("Dates:", len(F))

# --------------------------------------------------
# 2. x variables
# x[i,j,f] = 1 if team i hosts team j on date f
# --------------------------------------------------
x = {}

for i in T:
    for j in T:
        if i == j:
            continue
        for f in F:
            x[i, j, f] = model3.addVar(vtype=GRB.BINARY, name=f"x_{i}_{j}_{pd.Timestamp(f).strftime('%Y%m%d')}")

print(f"Sparse x variables created: {len(x)}")

# --------------------------------------------------
# 3. Basic schedule constraints
# (keep your original ones here if you already have them)
# Example placeholders:
#   - each team plays exactly one match on each played date
#   - home/away structure
#   - pairing constraints
# Replace / keep according to your existing Q1/Q2 model
# --------------------------------------------------

# Example: each team plays at most one game per date
for i in T:
    for f in F:
        model3.addConstr(
            gp.quicksum(x[i, j, f] for j in T if j != i) +
            gp.quicksum(x[j, i, f] for j in T if j != i)
            <= 1,
            name=f"one_game_{i}_{pd.Timestamp(f).strftime('%Y%m%d')}"
        )

# --------------------------------------------------
# 4. y variables
# y[i,f,z] = 1 if team i's game on date f is played in timezone z
# --------------------------------------------------
y = {}

for i in T:
    for f in played_dates[i]:
        for z in Z:
            y[i, f, z] = model3.addVar(vtype=GRB.BINARY, name=f"y_{i}_{pd.Timestamp(f).strftime('%Y%m%d')}_{z}")

print(f"y variables created: {len(y)}")

# --------------------------------------------------
# 5. Link y to x
# For each team i and played date f:
# exactly one timezone must be chosen, and it must match the arena timezone
# --------------------------------------------------
for i in T:
    for f in played_dates[i]:

        for z in Z:
            expr = gp.LinExpr()

            # if i is home vs j on f, game timezone = team_tz[i]
            expr += gp.quicksum(
                x[i, j, f] for j in T
                if j != i and (i, j, f) in x and team_tz[i] == z
            )

            # if i is away at j on f, game timezone = team_tz[j]
            expr += gp.quicksum(
                x[j, i, f] for j in T
                if j != i and (j, i, f) in x and team_tz[j] == z
            )

            model3.addConstr(
                y[i, f, z] == expr,
                name=f"link_{i}_{pd.Timestamp(f).strftime('%Y%m%d')}_{z}"
            )

        # exactly one timezone for that game date
        model3.addConstr(
            gp.quicksum(y[i, f, z] for z in Z) == 1,
            name=f"one_tz_{i}_{pd.Timestamp(f).strftime('%Y%m%d')}"
        )

# --------------------------------------------------
# 6. Build forbidden timezone triplets
# Forbidden if |z1-z2| + |z2-z3| >= 4
# --------------------------------------------------
bad_triplets = []
for z1 in Z:
    for z2 in Z:
        for z3 in Z:
            if abs(z1 - z2) + abs(z2 - z3) >= 4:
                bad_triplets.append((z1, z2, z3))

print(f"Bad timezone triplets: {len(bad_triplets)}")

# --------------------------------------------------
# 7. Q3 constraint
# For every team and every 3 consecutive matches:
# forbid bad timezone sequences
# --------------------------------------------------
q3_count = 0

for i in T:
    dates_i = sorted(played_dates[i])

    for k in range(len(dates_i) - 2):
        f1, f2, f3 = dates_i[k], dates_i[k+1], dates_i[k+2]

        for (z1, z2, z3) in bad_triplets:
            model3.addConstr(
                y[i, f1, z1] + y[i, f2, z2] + y[i, f3, z3] <= 2,
                name=f"tz_forbid_{i}_{k}_{z1}_{z2}_{z3}"
            )
            q3_count += 1

print(f"Q3 timezone constraints added: {q3_count}")

# --------------------------------------------------
# 8. Feasibility only
# --------------------------------------------------
model3.setObjective(0, GRB.MINIMIZE)

print("Starting optimization for Q3...")
model3.optimize()

print(f"Model status: {model3.status}")
print(f"NumVars: {model3.NumVars}")
print(f"NumConstrs: {model3.NumConstrs}")

# --------------------------------------------------
# 9. If infeasible, compute IIS
# --------------------------------------------------
if model3.status == GRB.INFEASIBLE:
    print("Model is infeasible. Computing IIS...")
    model3.computeIIS()
    model3.write("q3_infeasible.ilp")
    print("IIS written to q3_infeasible.ilp")

# --------------------------------------------------
# 10. Extract solution if feasible
# --------------------------------------------------
if model3.status == GRB.OPTIMAL:
    print("Feasible solution found for Question 3.")

    schedule_q3 = []
    for (i, j, f), var in x.items():
        if var.X > 0.5:
            schedule_q3.append({
                "Date": pd.Timestamp(f).strftime("%a, %b %d, %Y"),
                "Date_dt": pd.Timestamp(f),
                "Home": i,
                "Away": j,
                "Home_TZ": team_tz[i],
                "Away_TZ": team_tz[j],
                "Arena_TZ": team_tz[i]
            })

    schedule_q3_df = pd.DataFrame(schedule_q3).sort_values(["Date_dt", "Home"]).reset_index(drop=True)

    print(f"Generated schedule: {len(schedule_q3_df)} games")
    print(schedule_q3_df[["Date", "Home", "Away", "Arena_TZ"]].head(20).to_string(index=False))

    schedule_q3_df.to_csv("schedule_q3.csv", index=False)
    print("Saved to schedule_q3.csv")

else:
    print("No feasible solution found for Q3 under the additional timezone constraint.")

Set parameter OutputFlag to value 1
Teams: 16
Dates: 16
Sparse x variables created: 3840
y variables created: 1024
Bad timezone triplets: 14
Q3 timezone constraints added: 3136
Starting optimization for Q3...
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 23.3.0 23D2057)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 4672 rows, 4864 columns and 26816 nonzeros
Model fingerprint: 0xcef35704
Variable types: 0 continuous, 4864 integer (4864 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]
Found heuristic solution: objective 0.0000000

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 1: 0 

Optimal solution found (tolerance 1.00e-04)
Best objective 0.000000000000e+00, best bound 0.000000000000

In [9]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from collections import defaultdict

# =========================================================
# 0. Input assumptions
# ---------------------------------------------------------
# You must already have:
#   1) nba_sc : DataFrame with columns ['Date', 'Home', 'Visitor']
#   2) team_tz: dict, e.g. {'Atlanta Hawks': 0, ...}
# =========================================================

# Make sure Date is datetime
nba_sc = nba_sc.copy()
nba_sc["Date"] = pd.to_datetime(nba_sc["Date"])

required_cols = {"Date", "Home", "Visitor"}
missing = required_cols - set(nba_sc.columns)
if missing:
    raise ValueError(f"nba_sc missing columns: {missing}")

# =========================================================
# 1. Sets and basic data from original schedule
# =========================================================
T = sorted(set(nba_sc["Home"]).union(set(nba_sc["Visitor"])))
F = sorted(nba_sc["Date"].unique())

# timezone set
Z = sorted(set(team_tz[t] for t in T))

print("Teams:", len(T))
print("Dates:", len(F))
print("Timezones:", Z)

# For each date, which teams are originally home / away
home_teams_on_date = {
    f: sorted(nba_sc.loc[nba_sc["Date"] == f, "Home"].tolist())
    for f in F
}
away_teams_on_date = {
    f: sorted(nba_sc.loc[nba_sc["Date"] == f, "Visitor"].tolist())
    for f in F
}

# For each team, dates they play (fixed match sequence dates)
played_dates = {}
for t in T:
    dates_t = sorted(
        nba_sc.loc[
            (nba_sc["Home"] == t) | (nba_sc["Visitor"] == t),
            "Date"
        ].unique()
    )
    played_dates[t] = dates_t

# Original directed matchup counts:
# req_pair[(i,j)] = number of times i hosts j in the original schedule
req_pair = (
    nba_sc.groupby(["Home", "Visitor"])
    .size()
    .to_dict()
)

# Sanity checks
for f in F:
    h = home_teams_on_date[f]
    a = away_teams_on_date[f]
    if len(h) != len(a):
        raise ValueError(f"Date {f}: #home != #away")
    if set(h).intersection(set(a)):
        raise ValueError(f"Date {f}: some team is both home and away")

# =========================================================
# 2. Build model
# =========================================================
model3 = gp.Model("Q3_full")
model3.setParam("OutputFlag", 1)

# =========================================================
# 3. Sparse x variables
# x[i,j,f] = 1 if on date f, home team i hosts away team j
#
# IMPORTANT:
# We only create x for dates where i is one of the original
# home slots and j is one of the original away slots.
# This preserves the original home/away calendar structure.
# =========================================================
x = {}

for f in F:
    for i in home_teams_on_date[f]:
        for j in away_teams_on_date[f]:
            if i != j:
                x[i, j, f] = model3.addVar(
                    vtype=GRB.BINARY,
                    name=f"x_{i}_{j}_{pd.Timestamp(f).strftime('%Y%m%d')}"
                )

model3.update()
print(f"Sparse x variables created: {len(x)}")

# =========================================================
# 4. Basic schedule constraints
# ---------------------------------------------------------
# (A) Every home slot on every date hosts exactly one away team
# (B) Every away slot on every date visits exactly one home team
# (C) Preserve original directed matchup counts
# =========================================================

# (A) every home team/date gets exactly one opponent
for f in F:
    for i in home_teams_on_date[f]:
        model3.addConstr(
            gp.quicksum(x[i, j, f] for j in away_teams_on_date[f] if i != j) == 1,
            name=f"home_slot_{i}_{pd.Timestamp(f).strftime('%Y%m%d')}"
        )

# (B) every away team/date gets exactly one opponent
for f in F:
    for j in away_teams_on_date[f]:
        model3.addConstr(
            gp.quicksum(x[i, j, f] for i in home_teams_on_date[f] if i != j) == 1,
            name=f"away_slot_{j}_{pd.Timestamp(f).strftime('%Y%m%d')}"
        )

# (C) preserve directed pair frequencies from original schedule
for i in T:
    for j in T:
        if i == j:
            continue
        rhs = req_pair.get((i, j), 0)

        lhs_terms = []
        for f in F:
            if (i, j, f) in x:
                lhs_terms.append(x[i, j, f])

        model3.addConstr(
            gp.quicksum(lhs_terms) == rhs,
            name=f"pairfreq_{i}_{j}"
        )

# =========================================================
# 5. y variables
# y[i,f,z] = 1 if team i's match on date f is played in timezone z
# Since every team has exactly one match on each played date,
# exactly one timezone should be selected for that date.
# =========================================================
y = {}

for i in T:
    for f in played_dates[i]:
        for z in Z:
            y[i, f, z] = model3.addVar(
                vtype=GRB.BINARY,
                name=f"y_{i}_{pd.Timestamp(f).strftime('%Y%m%d')}_{z}"
            )

model3.update()
print(f"y variables created: {len(y)}")

# =========================================================
# 6. Link y to x
# ---------------------------------------------------------
# If team i is home on f -> arena timezone = team_tz[i]
# If team i is away at j on f -> arena timezone = team_tz[j]
# =========================================================
for i in T:
    for f in played_dates[i]:
        for z in Z:
            expr = gp.LinExpr()

            # i is home on f
            if i in home_teams_on_date[f] and team_tz[i] == z:
                expr += gp.quicksum(
                    x[i, j, f]
                    for j in away_teams_on_date[f]
                    if i != j and (i, j, f) in x
                )

            # i is away on f, so arena timezone = home team's timezone
            if i in away_teams_on_date[f]:
                expr += gp.quicksum(
                    x[j, i, f]
                    for j in home_teams_on_date[f]
                    if j != i and (j, i, f) in x and team_tz[j] == z
                )

            model3.addConstr(
                y[i, f, z] == expr,
                name=f"link_{i}_{pd.Timestamp(f).strftime('%Y%m%d')}_{z}"
            )

        # exactly one timezone on each played date
        model3.addConstr(
            gp.quicksum(y[i, f, z] for z in Z) == 1,
            name=f"one_tz_{i}_{pd.Timestamp(f).strftime('%Y%m%d')}"
        )

# =========================================================
# 7. Q3 additional constraint
# ---------------------------------------------------------
# No team should play 3 consecutive matches where
# |tz1 - tz2| + |tz2 - tz3| >= 4
# =========================================================
bad_triplets = []
for z1 in Z:
    for z2 in Z:
        for z3 in Z:
            if abs(z1 - z2) + abs(z2 - z3) >= 4:
                bad_triplets.append((z1, z2, z3))

print(f"Bad timezone triplets: {len(bad_triplets)}")

q3_count = 0
for i in T:
    dates_i = sorted(played_dates[i])

    # consecutive matches = consecutive entries in that team's played_dates
    for k in range(len(dates_i) - 2):
        f1, f2, f3 = dates_i[k], dates_i[k + 1], dates_i[k + 2]

        for (z1, z2, z3) in bad_triplets:
            model3.addConstr(
                y[i, f1, z1] + y[i, f2, z2] + y[i, f3, z3] <= 2,
                name=f"tz_forbid_{i}_{k}_{z1}_{z2}_{z3}"
            )
            q3_count += 1

print(f"Q3 timezone constraints added: {q3_count}")

# =========================================================
# 8. Feasibility only
# =========================================================
model3.setObjective(0, GRB.MINIMIZE)

print("Starting optimization for Q3...")
model3.optimize()

print(f"Model status: {model3.status}")
print(f"NumVars: {model3.NumVars}")
print(f"NumConstrs: {model3.NumConstrs}")

# =========================================================
# 9. If infeasible, compute IIS
# =========================================================
if model3.status == GRB.INFEASIBLE:
    print("No feasible solution found. Computing IIS...")
    model3.computeIIS()
    model3.write("q3_infeasible.ilp")
    print("IIS written to q3_infeasible.ilp")

# =========================================================
# 10. Extract solution
# =========================================================
if model3.status == GRB.OPTIMAL:
    print("Feasible solution found for Question 3.")

    schedule_q3 = []
    for (i, j, f), var in x.items():
        if var.X > 0.5:
            schedule_q3.append({
                "Date": pd.Timestamp(f).strftime("%a, %b %d, %Y"),
                "Date_dt": pd.Timestamp(f),
                "Home": i,
                "Away": j,
                "Home_TZ": team_tz[i],
                "Away_TZ": team_tz[j],
                "Arena_TZ": team_tz[i]   # game is played at home arena
            })

    schedule_q3_df = pd.DataFrame(schedule_q3).sort_values(
        ["Date_dt", "Home", "Away"]
    ).reset_index(drop=True)

    print(f"Generated schedule: {len(schedule_q3_df)} games")
    print(schedule_q3_df[["Date", "Home", "Away", "Arena_TZ"]].head(20).to_string(index=False))

    schedule_q3_df.to_csv("schedule_q3.csv", index=False)
    print("Saved to schedule_q3.csv")

else:
    print("No feasible solution found for Q3 under the additional timezone constraint.")

# =========================================================
# 11. Validation helpers
# =========================================================
def validate_solution(schedule_df, original_df, team_tz):
    out = {}

    temp = schedule_df.copy()
    temp["Date_dt"] = pd.to_datetime(temp["Date_dt"])

    # A) repeated directed matchups vs original counts
    sol_pair = temp.groupby(["Home", "Away"]).size().to_dict()
    orig_pair = original_df.groupby(["Home", "Visitor"]).size().to_dict()

    bad_pair = []
    all_pairs = set(sol_pair.keys()).union(set(orig_pair.keys()))
    for p in sorted(all_pairs):
        if sol_pair.get(p, 0) != orig_pair.get(p, 0):
            bad_pair.append({
                "Home": p[0],
                "Away": p[1],
                "Solution_Count": sol_pair.get(p, 0),
                "Original_Count": orig_pair.get(p, 0)
            })
    out["pair_count_diff"] = pd.DataFrame(bad_pair)

    # B) teams playing more than once per day
    home_rows = temp[["Date_dt", "Home"]].rename(columns={"Home": "Team"})
    away_rows = temp[["Date_dt", "Away"]].rename(columns={"Away": "Team"})
    team_games = pd.concat([home_rows, away_rows], ignore_index=True)

    multi_same_day = (
        team_games.groupby(["Date_dt", "Team"])
        .size()
        .reset_index(name="games_that_day")
        .query("games_that_day > 1")
    )
    out["multi_same_day"] = multi_same_day

    # C) Q3 violations
    home_sched = temp[["Date_dt", "Home", "Arena_TZ"]].rename(columns={"Home": "Team"})
    away_sched = temp[["Date_dt", "Away", "Arena_TZ"]].rename(columns={"Away": "Team"})
    team_sched = pd.concat([home_sched, away_sched], ignore_index=True).sort_values(["Team", "Date_dt"])

    violations = []
    for team, g in team_sched.groupby("Team"):
        g = g.sort_values("Date_dt").reset_index(drop=True)
        tzs = g["Arena_TZ"].tolist()
        dates = g["Date_dt"].tolist()

        for k in range(len(g) - 2):
            z1, z2, z3 = tzs[k], tzs[k + 1], tzs[k + 2]
            score = abs(z1 - z2) + abs(z2 - z3)
            if score >= 4:
                violations.append({
                    "Team": team,
                    "Date1": dates[k],
                    "Date2": dates[k + 1],
                    "Date3": dates[k + 2],
                    "TZ1": z1,
                    "TZ2": z2,
                    "TZ3": z3,
                    "SumAbsDiff": score
                })

    out["q3_violations"] = pd.DataFrame(violations)

    return out


# Optional validation run
if model3.status == GRB.OPTIMAL:
    checks = validate_solution(schedule_q3_df, nba_sc, team_tz)

    print("\nPair-count differences:")
    print(checks["pair_count_diff"] if not checks["pair_count_diff"].empty else "None")

    print("\nTeams playing more than once on same day:")
    print(checks["multi_same_day"] if not checks["multi_same_day"].empty else "None")

    print("\nQ3 timezone violations:")
    print(checks["q3_violations"] if not checks["q3_violations"].empty else "None")

Teams: 16
Dates: 16
Timezones: [0, 1, 2, 3]
Set parameter OutputFlag to value 1
Sparse x variables created: 1024
y variables created: 1024
Bad timezone triplets: 14
Q3 timezone constraints added: 3136
Starting optimization for Q3...
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 23.3.0 23D2057)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 4912 rows, 2048 columns and 16576 nonzeros
Model fingerprint: 0xa682aa89
Variable types: 0 continuous, 2048 integer (2048 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]
Presolve removed 2102 rows and 1467 columns
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -
Model s

In [12]:
if model3.status == GRB.INFEASIBLE:
    model3.computeIIS()
    print("\nConstraints in IIS:")
    for c in model3.getConstrs():
        if c.IISConstr:
            print(c.ConstrName)

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 23.3.0 23D2057)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads


IIS computed: 10 constraints, 0 bounds
IIS runtime: 0.00 seconds (0.00 work units)

Constraints in IIS:
home_slot_Los Angeles Lakers_20251111
home_slot_Los Angeles Lakers_20251115
pairfreq_Phoenix Suns_Los Angeles Lakers
link_Los Angeles Lakers_20251111_3
link_Los Angeles Lakers_20251113_2
link_Los Angeles Lakers_20251113_3
one_tz_Los Angeles Lakers_20251113
link_Los Angeles Lakers_20251115_3
tz_forbid_Los Angeles Lakers_4_3_0_3
tz_forbid_Los Angeles Lakers_4_3_1_3


## We formulated Question 3 by preserving all prior scheduling constraints and adding the requirement that for any team, the sum of timezone jumps across any three consecutive matches must be less than 4. Gurobi returned an infeasible status. We further computed an IIS, which confirmed that the infeasibility is structural rather than caused by a coding or solver issue. Therefore, we conclude that no feasible schedule exists under the additional timezone constraint.